# 02 名称对齐、单位整理与独立实验记录

目标：把上一环节的 CSV 变成可关联的记录；明确“同一化合物”和“同一次实验”是两种不同身份。

本环节是对 ReactionSeek 标准化思路的**有界教学适配**：使用有来源的 4 个化合物身份缓存、RDKit 和明确的单位规则，不声称复现了原作者完整的在线 PubChem/OPSIN/LLM 匹配流程。未知名称保持未解析，不强制匹配最近实体。

先完成 01。本 Notebook 不调用 API、不联网。

In [ ]:
from pathlib import Path
import os
import sys
import json

# 从仓库根目录或 notebooks/ 打开均可；不依赖个人机器的绝对路径。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/reaction_knowledge_lab").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请在 reaction-knowledge-lab 仓库根目录或 notebooks 目录启动。")
sys.path.insert(0, str(ROOT / "src"))
run_setting = Path(os.environ.get("RKL_NOTEBOOK_RUN_DIR", "runs/notebook-lab")).expanduser()
LAB = run_setting.resolve() if run_setting.is_absolute() else (ROOT / run_setting).resolve()
LAB.mkdir(parents=True, exist_ok=True)
RAW, ALIGNED, DB = LAB / "extracted.csv", LAB / "aligned.json", LAB / "knowledge.sqlite"
from reaction_knowledge_lab.files import DATA, read_json, write_json, load_records, save_records
print("仓库：", ROOT)
print("本次实验目录：", LAB)
print("Python：", sys.executable)

In [ ]:
if not RAW.is_file():
    raise FileNotFoundError(f"请先完成 01，预期文件：{RAW}")
if DB.exists():
    raise FileExistsError("这个实验目录已建库；重做请使用新的 RKL_NOTEBOOK_RUN_DIR。")
from reaction_knowledge_lab.alignment import (
    entity_cache, resolve_alias, time_minutes, temperature_range, parse_amount, align_csv,
)

## 1. 同物异名能否链接到同一个对象

先比较乙腈的名称与缩写，再比较二苯甲酮的两个名称。这个小缓存用于教学核对，不意味着所有同义词或任意化合物都已覆盖。

In [ ]:
for first, second in [("acetonitrile", "CH3CN"), ("benzophenone", "diphenylmethanone")]:
    a, b = resolve_alias(first), resolve_alias(second)
    print(first, "↔", second, ":", a["entity_id"], a["smiles"])
    assert a["entity_id"] == b["entity_id"]
    assert a["smiles"] == b["smiles"]
print("缓存实体数：", len(entity_cache()))

In [ ]:
unknown = resolve_alias("unspecified aromatic substrate")
print(json.dumps(unknown, ensure_ascii=False, indent=2))
assert unknown["resolution_status"] == "unresolved"
assert unknown["entity_id"] is None and unknown["smiles"] is None

## 2. 只进行有依据的单位换算

36 小时可以精确换算成 2160 分钟；`overnight` 没有给出精确时长，必须保留原词与空值。30–35 ℃保留为区间，不取平均数，不冒称恒温设定值。用量优先使用原文明确报告的 mmol，质量原值仍留在原始记录中。

In [ ]:
unit_examples = {
    "36 hours": time_minutes("36 hours"),
    "overnight": time_minutes("overnight"),
    "30–35°C": temperature_range("30–35°C"),
    "room temperature": temperature_range("room temperature"),
    "3.61 g (20.0 mmol)": parse_amount("3.61 g (20.0 mmol)"),
}
print(json.dumps(unit_examples, ensure_ascii=False, indent=2))
assert unit_examples["36 hours"] == 2160
assert unit_examples["overnight"] is None
assert unit_examples["30–35°C"] == {"min": 30.0, "max": 35.0}
assert unit_examples["room temperature"] is None

## 3. 整理主案例 CSV

输入文件来自 01；输出包括结构化记录与原始名称到统一实体的映射。实验 ID 暂按这篇文献的表格行生成，下一环节必须用原文核对；它不是按反应物/产物结构去重产生的。

In [ ]:
if ALIGNED.exists():
    raise FileExistsError(f"已有对齐结果：{ALIGNED}。查看现有文件，或换新目录从 01 重做。")
summary = align_csv(RAW, ALIGNED)
records = load_records(ALIGNED)
mappings = read_json(ALIGNED.with_suffix(".mapping.json"))["mappings"]
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(mappings, ensure_ascii=False, indent=2))
assert len(records) == 1 and summary["unresolved_names"] == []
assert len(records[0]["inputs"]) == 2  # 化合物名称中的逗号没有导致错误拆分。

In [ ]:
r1 = records[0]
print("实验：", r1["record_id"], "原始行：", r1["raw_record_id"])
print("产物：", r1["product"])
print("条件：", r1["conditions"])
print("来源：", r1["source"])
print("未改写的原值：", r1["raw"])
assert r1["conditions"]["time_minutes"] == 2160
assert r1["conditions"]["temperature_c"] == {"min": 30.0, "max": 35.0}
assert r1["raw"]["Reaction time"] == "36 hours"
assert r1["raw"]["Yield"] == "72%"

## 4. 检查粒度：共享对象，保留不同实验

此时知识资源仍只有抽取出来的第一条实验。对齐步骤不能凭空补出第二次实验，也不能因为两次产物相同而将它们合并。我们先查看课程参考中的身份关系，真正补录要到 03 的有证据修订步骤完成。

In [ ]:
reference_records = read_json(DATA / "reference/case22.json")["records"]
assert len({r["record_id"] for r in reference_records}) == 2
assert len({r["product"]["entity_id"] for r in reference_records}) == 1
print("参考中的不同实验ID：", [r["record_id"] for r in reference_records])
print("共享的产物实体ID：", reference_records[0]["product"]["entity_id"])
print("当前对齐文件中的实验数：", len(records))
write_json(LAB / "alignment_checks.json", {
    "alias_identity_shared": resolve_alias("CH3CN")["entity_id"] == resolve_alias("acetonitrile")["entity_id"],
    "unknown_not_guessed": unknown["entity_id"] is None,
    "range_retained": r1["conditions"]["temperature_c"],
    "record_count_after_alignment": len(records), "raw_record_id_retained": r1["raw_record_id"],
})

**提交内容：** `aligned.json`、`aligned.mapping.json`、`alignment_checks.json`。

解释两个问题：为什么统一化合物 ID 后，仍需要独立的实验 ID？为什么 `overnight` 不能直接写成 720 分钟？完成后打开 `03_validation_update.ipynb`。